# 01. Environment & Handicap Setup

## CS377 Reinforcement Learning — Team 1
### Concentration vs. Dispersion in Free-Setup Chess via AlphaZero

---

This notebook contains the **experimental environment definition**:

1. **Configuration Loader** — YAML 기반 패턴/하이퍼파라미터 설정
2. **Handicap Pattern & FEN Generator** — 7개 removal 패턴 정의, 양측 30pt 핸디캡 보드 생성
3. **Game Log Schema & I/O** — JSONL 기반 게임 기록 스키마

#### Removal Patterns (9 points each)

| Pattern | Score | Pieces Removed | Concentration |
|---------|-------|---------------|--------------|
| Queen | 9 | 1 | Highest |
| Rook + Bishop + Pawn | 5+3+1 | 3 | High |
| Rook + Knight + Pawn | 5+3+1 | 3 | High |
| Bishop + Bishop + Knight | 3+3+3 | 3 | High |
| Rook + 4 Pawns | 5+4 | 5 | Medium |
| Bishop + Knight + 3 Pawns | 3+3+3 | 5 | Medium |
| Bishop + 6 Pawns | 3+6 | 7 | Low |
| Knight + 6 Pawns | 3+6 | 7 | Low |

---
## 1. Configuration Loader

YAML 기반의 패턴 정의와 하이퍼파라미터를 로드합니다.  
- `config/patterns.yaml` — 7개 removal 패턴 정의  
- `config/default.yaml` — 학습/MCTS/Arena 하이퍼파라미터  

Source: `handichess/config/__init__.py`

In [ ]:
"""Configuration loading and pattern definitions."""

import os
import yaml
from pathlib import Path

CONFIG_DIR = Path("config")  # Submit_Codes/config/


def load_yaml(filename: str) -> dict:
    """Load a YAML config file from the config directory."""
    filepath = CONFIG_DIR / filename
    with open(filepath, "r") as f:
        return yaml.safe_load(f)


def load_patterns() -> dict:
    """Load removal pattern configurations."""
    return load_yaml("patterns.yaml")


def load_defaults() -> dict:
    """Load default hyperparameters."""
    return load_yaml("default.yaml")

---
## 2. Matchup Position Generator (Design B)

각 패턴은 **NoQ vs Q** 매치업으로 설계됩니다:
- **NoQ side**: Queen(9pt)을 제거
- **Q side**: Bundle(9pt 합계)을 제거
- 양측 모두 정확히 **30 material points**로 시작

Source: `handichess/common/handicap.py`

In [ ]:
"""
Matchup position generator (Design B).

Generates FEN strings for NoQ vs Q match-ups.
Each pattern specifies 9 points of material removed for both sides:
- NoQ side: removes Queen (9 pts).
- Q side: removes a Bundle (9 pts).
Both sides end up with exactly 30 points of material.
"""

from __future__ import annotations

import chess
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import yaml


# ── Standard piece point values ──────────────────────────────────────────────
PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
}

# Map single-letter piece codes (from YAML) → python-chess piece types
_PIECE_CODE_MAP = {
    "P": chess.PAWN,
    "N": chess.KNIGHT,
    "B": chess.BISHOP,
    "R": chess.ROOK,
    "Q": chess.QUEEN,
    "K": chess.KING,
}


# ── Data classes ─────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Removal:
    """A single piece removal: piece type + square name (for white side)."""
    piece: int            # chess.PAWN … chess.QUEEN
    square_name: str      # e.g. "d1", "a2"

    @property
    def white_square(self) -> int:
        """Square index when the piece is removed from white."""
        return chess.parse_square(self.square_name)

    @property
    def black_square(self) -> int:
        """Mirror square for when the piece is removed from black (flip rank)."""
        sq = self.white_square
        return chess.square_mirror(sq)


@dataclass(frozen=True)
class MatchupPattern:
    """A complete match-up pattern (both sides remove 9 points)."""
    pattern_id: str
    description: str
    total_points: int      # Usually 9
    num_removed: int       # Number of pieces in the bundle
    concentration: str     # "highest" | "high" | "medium" | "low"
    phase: int             # 1 or 2
    noq_removals: tuple[Removal, ...]  # Always Queen
    q_removals: tuple[Removal, ...]    # The Bundle

    def bundle_vector(self) -> dict[str, int]:
        """
        Returns the count of each piece type in the Bundle (q_removals).
        Keys: "Q", "R", "B", "N", "P".
        """
        counts = {"Q": 0, "R": 0, "B": 0, "N": 0, "P": 0}
        code_map_inv = {v: k for k, v in _PIECE_CODE_MAP.items() if k != "K"}
        for r in self.q_removals:
            code = code_map_inv[r.piece]
            counts[code] += 1
        return counts


@dataclass
class MatchupPosition:
    """A concrete match-up starting position."""
    pattern_id: str
    noq_color: chess.Color       # chess.WHITE or chess.BLACK
    fen: str
    bundle_vector: dict[str, int]

    def to_dict(self) -> dict:
        return {
            "pattern_id": self.pattern_id,
            "noq_color": "white" if self.noq_color == chess.WHITE else "black",
            "fen": self.fen,
            "bundle_vector": self.bundle_vector,
        }


# ── Pattern loading ──────────────────────────────────────────────────────────
def _load_patterns_from_yaml(yaml_path: Optional[Path] = None) -> list[MatchupPattern]:
    """Load match-up patterns from the YAML config file."""
    if yaml_path is None:
        yaml_path = Path("config") / "patterns.yaml"

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    patterns = []
    for pid, pdata in data["patterns"].items():
        noq_rems = []
        for r in pdata.get("noq_removals", []):
            piece_type = _PIECE_CODE_MAP[r["piece"]]
            noq_rems.append(Removal(piece=piece_type, square_name=r["square"]))

        q_rems = []
        for r in pdata.get("q_removals", []):
            piece_type = _PIECE_CODE_MAP[r["piece"]]
            q_rems.append(Removal(piece=piece_type, square_name=r["square"]))

        pattern = MatchupPattern(
            pattern_id=pid,
            description=pdata["description"],
            total_points=pdata["total_points"],
            num_removed=pdata["num_removed"],
            concentration=pdata["concentration"],
            phase=pdata["phase"],
            noq_removals=tuple(noq_rems),
            q_removals=tuple(q_rems),
        )

        # Sanity check: both removals must sum to 9 points
        noq_sum = sum(PIECE_VALUES[r.piece] for r in noq_rems)
        q_sum = sum(PIECE_VALUES[r.piece] for r in q_rems)
        
        assert noq_sum == 9, f"Pattern '{pid}': NoQ removals sum to {noq_sum}, expected 9"
        assert q_sum == 9, f"Pattern '{pid}': Q removals sum to {q_sum}, expected 9"

        patterns.append(pattern)

    return patterns


# ── Module-level cache ───────────────────────────────────────────────────────
_PATTERNS: Optional[list[MatchupPattern]] = None


def get_patterns(phase: Optional[int] = None) -> list[MatchupPattern]:
    """
    Get all match-up patterns, optionally filtered by phase.
    """
    global _PATTERNS
    if _PATTERNS is None:
        _PATTERNS = _load_patterns_from_yaml()

    if phase is not None:
        return [p for p in _PATTERNS if p.phase == phase]
    return list(_PATTERNS)


def get_pattern_by_id(pattern_id: str) -> MatchupPattern:
    """Look up a single pattern by its ID."""
    for p in get_patterns():
        if p.pattern_id == pattern_id:
            return p
    raise ValueError(f"Unknown pattern: {pattern_id}")


# ── Utilities ────────────────────────────────────────────────────────────────
def count_material(board: chess.Board, color: chess.Color) -> int:
    """Count total material points for a side (excluding king)."""
    total = 0
    for piece_type in PIECE_VALUES:
        total += len(board.pieces(piece_type, color)) * PIECE_VALUES[piece_type]
    return total


def material_balance(board: chess.Board) -> int:
    """Material advantage for white (white - black), in standard points."""
    return count_material(board, chess.WHITE) - count_material(board, chess.BLACK)


# ── Position generation ──────────────────────────────────────────────────────
def make_matchup_board(
    pattern: MatchupPattern,
    noq_color: chess.Color,
) -> chess.Board:
    """
    Create a chess.Board for a NoQ vs Q match-up.

    Args:
        pattern: The MatchupPattern specifying noq_removals and q_removals.
        noq_color: Which color plays the NoQ side (removes noq_removals).
                   The opposite color plays the Q side (removes q_removals).

    Returns:
        A python-chess Board ready to play from.
    """
    board = chess.Board()  # standard starting position

    # Determine colors
    q_color = not noq_color

    # 1. Apply NoQ removals
    for removal in pattern.noq_removals:
        sq = removal.white_square if noq_color == chess.WHITE else removal.black_square
        piece = board.piece_at(sq)
        expected_piece = chess.Piece(removal.piece, noq_color)
        assert piece == expected_piece, (
            f"Pattern '{pattern.pattern_id}': expected {expected_piece} at "
            f"{chess.square_name(sq)}, found {piece}"
        )
        board.remove_piece_at(sq)

    # 2. Apply Q (Bundle) removals
    for removal in pattern.q_removals:
        sq = removal.white_square if q_color == chess.WHITE else removal.black_square
        piece = board.piece_at(sq)
        expected_piece = chess.Piece(removal.piece, q_color)
        assert piece == expected_piece, (
            f"Pattern '{pattern.pattern_id}': expected {expected_piece} at "
            f"{chess.square_name(sq)}, found {piece}"
        )
        board.remove_piece_at(sq)

    # 3. Validation: Both sides must have exactly 30 material points.
    assert count_material(board, chess.WHITE) == 30, "White material is not 30."
    assert count_material(board, chess.BLACK) == 30, "Black material is not 30."

    # Update castling rights
    board.set_castling_fen(_compute_castling_fen(board))

    return board


def _compute_castling_fen(board: chess.Board) -> str:
    """Recompute castling FEN based on actual rook/king positions."""
    castling = ""

    # White
    wk = board.king(chess.WHITE)
    if wk == chess.E1:
        if board.piece_at(chess.H1) == chess.Piece(chess.ROOK, chess.WHITE):
            castling += "K"
        if board.piece_at(chess.A1) == chess.Piece(chess.ROOK, chess.WHITE):
            castling += "Q"

    # Black
    bk = board.king(chess.BLACK)
    if bk == chess.E8:
        if board.piece_at(chess.H8) == chess.Piece(chess.ROOK, chess.BLACK):
            castling += "k"
        if board.piece_at(chess.A8) == chess.Piece(chess.ROOK, chess.BLACK):
            castling += "q"

    return castling if castling else "-"


def generate_position(
    pattern: MatchupPattern,
    noq_color: chess.Color,
) -> MatchupPosition:
    """
    Generate a single MatchupPosition for a given pattern and noq_color.
    """
    board = make_matchup_board(pattern, noq_color)
    return MatchupPosition(
        pattern_id=pattern.pattern_id,
        noq_color=noq_color,
        fen=board.fen(),
        bundle_vector=pattern.bundle_vector(),
    )


def generate_all_positions(
    phase: Optional[int] = None,
) -> list[MatchupPosition]:
    """
    Generate all match-up positions (both colors) for all patterns.
    """
    positions = []
    for pattern in get_patterns(phase=phase):
        for side in [chess.WHITE, chess.BLACK]:
            positions.append(generate_position(pattern, side))
    return positions

---
## 3. Game Log Schema & I/O

모든 실험 결과는 JSONL 형식의 `GameRecord`로 기록됩니다.  
`GameLog` 클래스가 append-only 쓰기, 읽기, DataFrame 변환, 필터링을 제공합니다.

Source: `handichess/common/gamelog.py`

In [ ]:
"""
Game log schema and I/O.

Each game is logged as a JSON object (one per line in a .jsonl file).
All analysis code consumes these logs exclusively.
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Optional, Iterator

import pandas as pd


@dataclass
class GameRecord:
    """
    Record of a single completed game.

    Fields:
        pattern_id:       Removal pattern identifier (e.g. "rook_bishop_pawn").
        q_side:           "white" or "black" — which side has the Queen.
        result:           Game result from the *q_side's* perspective:
                          "win", "draw", or "loss".
        result_score:     Numeric score: 1.0 (win), 0.5 (draw), 0.0 (loss).
        ply:              Total number of half-moves played.
        start_fen:        Starting FEN of the game.
        termination:      How the game ended: "checkmate", "stalemate",
                          "threefold", "fifty_moves", "insufficient",
                          "max_moves", "resignation".
        engine:           Engine/agent identifier (e.g. "lc0", "az_v3", "random").
        nodes:            Nodes/simulations per move (for reproducibility).
        timestamp:        ISO-format timestamp of game completion.
        extra:            Optional dict for additional metadata.
    """
    pattern_id: str
    q_side: str                 # "white" | "black"
    noq_side: str               # "white" | "black"
    result: str                 # "win" | "draw" | "loss"
    result_score: float         # 1.0 | 0.5 | 0.0
    ply: int
    start_fen: str
    termination: str
    engine: str
    nodes: int
    timestamp: str = ""
    extra: Optional[dict] = None

    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()
        # Validate
        assert self.q_side in ("white", "black"), (
            f"Invalid q_side: {self.q_side}"
        )
        assert self.noq_side in ("white", "black"), (
            f"Invalid noq_side: {self.noq_side}"
        )
        assert self.result in ("win", "draw", "loss"), (
            f"Invalid result: {self.result}"
        )
        assert self.result_score in (0.0, 0.5, 1.0), (
            f"Invalid result_score: {self.result_score}"
        )


def result_from_outcome(
    outcome: str,
    q_side: str,
) -> tuple[str, float]:
    """
    Convert a game outcome string to (result, result_score) from the
    Q side's perspective.
    """
    if outcome == "1/2-1/2":
        return "draw", 0.5
    elif outcome == "1-0":
        if q_side == "white":
            return "win", 1.0
        else:
            return "loss", 0.0
    elif outcome == "0-1":
        if q_side == "black":
            return "win", 1.0
        else:
            return "loss", 0.0
    else:
        raise ValueError(f"Unknown outcome: {outcome}")


# ── I/O ──────────────────────────────────────────────────────────────────────

class GameLog:
    """
    Append-only game log backed by a JSONL file.

    Usage:
        log = GameLog("runs/experiment_1/games.jsonl")
        log.write(record)
        for record in log.read():
            ...
        df = log.to_dataframe()
    """

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, record: GameRecord) -> None:
        """Append a single game record to the log file."""
        with open(self.path, "a") as f:
            d = asdict(record)
            if d.get("extra") is None:
                del d["extra"]
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

    def write_many(self, records: list[GameRecord]) -> None:
        """Append multiple game records at once."""
        with open(self.path, "a") as f:
            for record in records:
                d = asdict(record)
                if d.get("extra") is None:
                    del d["extra"]
                f.write(json.dumps(d, ensure_ascii=False) + "\n")

    def read(self) -> Iterator[GameRecord]:
        """Read all game records from the log file."""
        if not self.path.exists():
            return
        with open(self.path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                yield GameRecord(**d)

    def read_all(self) -> list[GameRecord]:
        """Read all records into a list."""
        return list(self.read())

    def to_dataframe(self) -> pd.DataFrame:
        """Convert the log to a pandas DataFrame for analysis."""
        records = self.read_all()
        if not records:
            return pd.DataFrame()
        return pd.DataFrame([asdict(r) for r in records])

    def count(self) -> int:
        """Count total number of games in the log."""
        if not self.path.exists():
            return 0
        with open(self.path, "r") as f:
            return sum(1 for line in f if line.strip())

    def filter(
        self,
        pattern_id: Optional[str] = None,
        q_side: Optional[str] = None,
        engine: Optional[str] = None,
    ) -> list[GameRecord]:
        """Filter records by pattern, side, or engine."""
        results = []
        for record in self.read():
            if pattern_id and record.pattern_id != pattern_id:
                continue
            if q_side and record.q_side != q_side:
                continue
            if engine and record.engine != engine:
                continue
            results.append(record)
        return results


def merge_logs(log_paths: list[str | Path], output_path: str | Path) -> GameLog:
    """Merge multiple log files into one."""
    output = GameLog(output_path)
    for path in log_paths:
        source = GameLog(path)
        for record in source.read():
            output.write(record)
    return output

---
## 4. Demo: 패턴별 FEN 생성 및 보드 시각화

In [ ]:
# Generate and display all matchup positions
print("=== Matchup Positions ===")
for pos in generate_all_positions():
    color_str = "WHITE" if pos.noq_color == chess.WHITE else "BLACK"
    print(f"[{pos.pattern_id}] NoQ_Color={color_str}")
    print(f"  FEN: {pos.fen}")
    print(f"  Bundle removed from Q-side: {pos.bundle_vector}")
    print()

# Verify material balance for all positions
print("\n=== Material Verification ===")
for pattern in get_patterns():
    for color in [chess.WHITE, chess.BLACK]:
        board = make_matchup_board(pattern, color)
        w_mat = count_material(board, chess.WHITE)
        b_mat = count_material(board, chess.BLACK)
        side_str = "WHITE" if color == chess.WHITE else "BLACK"
        print(f"{pattern.pattern_id} (NoQ={side_str}): W={w_mat}, B={b_mat} ✓" 
              if w_mat == 30 and b_mat == 30 
              else f"{pattern.pattern_id} (NoQ={side_str}): W={w_mat}, B={b_mat} ✗")

In [ ]:
# Visualize a sample board (requires jupyter notebook with SVG support)
import chess.svg
from IPython.display import SVG, display

pattern = get_pattern_by_id("rook_bishop_pawn")
board = make_matchup_board(pattern, chess.WHITE)
print(f"Pattern: {pattern.pattern_id} ({pattern.description})")
print(f"FEN: {board.fen()}")
display(SVG(chess.svg.board(board, size=400)))